# 📧 Email Marketing — Campaign & Lead Management

**Manages campaigns, email templates, and lead-to-campaign assignments via Notion.**

| Section | Functions | Databases |
|---------|-----------|----------|
| 1 — Read | `read_campaigns()`, `read_templates()`, `read_leads()` | campaigns, email_sequences, leads_crm |
| 2 — Campaigns | `create_campaign()`, `activate_campaign()`, `pause_campaign()` | campaigns |
| 3 — Templates | `create_template()`, `list_templates_for_campaign()` | email_sequences |
| 4 — Lead Assignment | `assign_leads_by_score()`, `preview_assignments()`, `clear_assignments()` | leads_crm |

### Property Access

| Database | Read + Write | Read Only |
|----------|-------------|----------|
| **campaigns** | Name, Status, Send Frequency, Priority, Goal, Start Date, Notes | Campaign Score, Open Rate, Click Rate, Total Sent, Bounce Rate, Reply Rate |
| **email_sequences** | Template, Subject Line, Campaign, Status, Trigger, Send Delay Hours, Notes | Open Rate, Click Rate, Unsubscribe Rate, Total Sent, Bounce Rate, Reply Rate |
| **leads_crm** | Audience relation only | Email, Name, First Name, Score, Segment, Status, Unsubscribed, Pipeline Stage, Last Touch, Source, Signals |

> **Note:** Email sending / send-planning is handled by [Send_Emails.ipynb](Send_Emails.ipynb). This notebook manages the data layer only.

In [ ]:
# ── Cell 1: Setup & Configuration ────────────────────────────────────────
import sys, os

_WS_ROOT = r"C:\Users\sossi\Desktop\Business\Orchestrator Hedge Edge"
_MARKETING_DIR = os.path.join(_WS_ROOT, "Business", "GROWTH", "executions", "Marketing")

# Ensure both workspace root and marketing package parent are importable
for p in [_WS_ROOT, _MARKETING_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

from dotenv import load_dotenv
load_dotenv(os.path.join(_WS_ROOT, ".env"))

from shared.notion_client import DATABASES
import shared.notion_client as _nc

# ── Notebook-safe patch: add relation handling once only ──
if not getattr(_nc, "_send_emails_relation_patch_applied", False):
    _nc._send_emails_base_extract = _nc._extract_value

    def _patched_extract(prop):
        if prop.get("type") == "relation":
            return [r["id"] for r in prop.get("relation", [])]
        return _nc._send_emails_base_extract(prop)

    _nc._extract_value = _patched_extract
    _nc._send_emails_relation_patch_applied = True
    print("  Applied relation patch for notebook reads")
else:
    print("  Relation patch already active")

# ── Import execution modules ──
from email_marketing.campaigns import read_campaigns, create_campaign, activate_campaign, pause_campaign
from email_marketing.templates import read_templates, create_template, list_templates_for_campaign
from email_marketing.leads import read_leads, preview_assignments, assign_leads_by_score, clear_assignments
from email_marketing.leads import _read_leads_full
from email_marketing.config import SEGMENTS

print("Setup complete")
print(f"  NOTION_API_KEY set: {bool(os.getenv('NOTION_API_KEY') or os.getenv('NOTION_TOKEN'))}")
print(f"  Campaigns DB:       {DATABASES['campaigns'][:8]}...")
print(f"  Email Sequences DB: {DATABASES['email_sequences'][:8]}...")
print(f"  Leads CRM DB:       {DATABASES['leads_crm'][:8]}...")
print(f"  Functions imported:  campaigns(4) templates(3) leads(4)")

## Section 1 — Notion Read

Read-only queries against all three databases. Returns clean Python dicts with both writable and read-only fields.

In [ ]:
# ── Section 1: Read Functions ─────────────────────────────────────────
#   read_campaigns(status_filter=None)
#   read_templates(campaign_id=None)
#   read_leads(segment=None, exclude_unsubscribed=True)

print("Read functions imported: read_campaigns(), read_templates(), read_leads()")

In [ ]:
# ── Section 1 Demo: Read all three databases ─────────────────────

from shared.notion_client import notion_request, _API_BASE

# Campaigns
campaigns = read_campaigns()
print(f"=== CAMPAIGNS ({len(campaigns)}) ===")
for c in campaigns:
    metrics = f"score={c['campaign_score']}  open={c['open_rate']}  click={c['click_rate']}  bounce={c['bounce_rate']}  reply={c['reply_rate']}"
    print(f"  [{c['status']:>15}] {c['name']:<35} freq={c['send_frequency']}h  {metrics}")

# Templates (all)
templates = read_templates()
print(f"\n=== EMAIL TEMPLATES ({len(templates)}) ===")
for t in templates:
    metrics = f"open={t['open_rate']}  click={t['click_rate']}  bounce={t['bounce_rate']}  reply={t['reply_rate']}"
    print(f"  {t['template_name']:<40} subject: {t['subject_line'][:40]}  {metrics}")

# Leads preview (first page only — avoids slow full-table scan)
print("\n=== LEADS PREVIEW (first 10 rows) ===")
resp = notion_request(
    "post",
    f"{_API_BASE}/databases/{DATABASES['leads_crm']}/query",
    headers={
        "Authorization": f"Bearer {os.getenv('NOTION_API_KEY') or os.getenv('NOTION_TOKEN')}",
        "Notion-Version": "2022-06-28",
        "Content-Type": "application/json",
    },
    json={"page_size": 10},
    timeout=60,
)
resp.raise_for_status()
data = resp.json()
for page in data.get("results", []):
    props = page.get("properties", {})
    email = props.get("Email", {}).get("email") or ""
    first_name = "".join(item.get("plain_text", "") for item in props.get("First Name", {}).get("rich_text", []))
    source_prop = props.get("Source", {})
    source = (source_prop.get("select") or {}).get("name", "")
    unsubscribed = props.get("Unsubscribed", {}).get("checkbox", False)
    print(f"  {email:<35} first_name={first_name or '-':<12} unsub={unsubscribed}  src={source}")

print(f"\nPreviewed {len(data.get('results', []))} leads from leads_crm (first page only)")

## Section 2 — Campaign Management

Create, activate, and pause campaigns. Writes to: **Name, Status, Send Frequency, Priority, Goal, Start Date, Notes**.

In [ ]:
# ── Section 2: Campaign Management Functions ─────────────────────
#   create_campaign(name, send_frequency=48, priority="Medium", goal="", notes="")
#   activate_campaign(campaign_id, campaign_name="")
#   pause_campaign(campaign_id, campaign_name="")

print("Campaign functions imported: create_campaign(), activate_campaign(), pause_campaign()")

In [ ]:
# ── Section 2 Demo: Campaign lifecycle ────────────────────────────
# Uncomment to test (writes to Notion):

# new = create_campaign(
#     name="Q1 Prop Trading Nurture",
#     send_frequency=72,
#     goal="Convert warm leads from prop trading segment",
#     campaign_score=50,
# )

# activate_campaign(new["id"], new["name"])
# pause_campaign(new["id"], new["name"])

# Verify current campaigns
print("Current campaigns:")
for c in read_campaigns():
    print(f"  [{c['status']:>15}] {c['name']}")

## Section 3 — Email Templates

Create email templates (sequences) linked to campaigns. Writes to: **Template, Subject Line, Campaign (relation), Status, Trigger, Send Delay Hours, Notes**.

In [ ]:
# ── Section 3: Email Template Functions ───────────────────────────
#   create_template(campaign_id, template_name, subject_line, ...)
#   list_templates_for_campaign(campaign_id)

print("Template functions imported: create_template(), list_templates_for_campaign()")

In [ ]:
# ── Section 3 Demo: List templates per active campaign ────────────────

active = read_campaigns(status_filter="Active")
if active:
    for camp in active:
        cid = camp["id"]
        print(f"\nTemplates for '{camp['name']}':")
        tpls = list_templates_for_campaign(cid)
        if tpls:
            for t in tpls:
                print(f"  #{t['_seq_num']}  {t['template_name']:<35} subject: {t['subject_line'][:50]}")
                print(f"       open={t['open_rate']}  click={t['click_rate']}  bounce={t['bounce_rate']}  reply={t['reply_rate']}")
        else:
            print("  (no templates)")
else:
    print("No active campaigns — create and activate one in Section 2 first.")

## Section 4 — Lead Assignment

Assign leads to campaigns via the `Campaign` dual-relation. This is the **only** writable property on leads_crm.

Score thresholds: **Cold** (0–29), **Warm** (30–59), **Hot** (60+)

In [ ]:
# ── Section 4: Lead Assignment Functions ──────────────────────────
#   preview_assignments(campaign_id, min_score=0, segment=None)
#   assign_leads_by_score(campaign_id, min_score=0, segment=None, dry_run=True)
#   clear_assignments(campaign_id, dry_run=True)

print("Lead functions imported: preview_assignments(), assign_leads_by_score(), clear_assignments()")

In [ ]:
# ── Section 4 Demo: Which leads are in which campaigns? ───────────────
# Full view: load all leads with campaign relations, cross-reference campaign names

leads_all, email_to_rel = _read_leads_full(exclude_unsubscribed=True)
campaigns_all = read_campaigns()
camp_id_to_name = {c["id"]: c["name"] for c in campaigns_all}

# Build per-campaign lead lists
campaign_leads = {}
unassigned = []
for lead in leads_all:
    rel_ids = email_to_rel.get(lead["email"], [])
    if not rel_ids:
        unassigned.append(lead)
    else:
        for cid in rel_ids:
            cname = camp_id_to_name.get(cid, cid[:8] + "...")
            campaign_leads.setdefault(cname, []).append(lead)

print("=" * 70)
print("  LEAD → CAMPAIGN ASSIGNMENTS")
print("=" * 70)
for cname, members in sorted(campaign_leads.items(), key=lambda x: -len(x[1])):
    print(f"\n  📧 {cname}  ({len(members)} leads)")
    for m in members[:5]:
        print(f"      {m['email']:<35} [{m['segment']}] score={m['score']}")
    if len(members) > 5:
        print(f"      ... and {len(members) - 5} more")

print(f"\n  ⚠️  Unassigned leads: {len(unassigned)}")
for m in unassigned[:5]:
    print(f"      {m['email']:<35} [{m['segment']}] score={m['score']}")
if len(unassigned) > 5:
    print(f"      ... and {len(unassigned) - 5} more")

# Summary
total_assigned = sum(len(v) for v in campaign_leads.values())
print(f"\n{'─' * 70}")
print(f"  Total leads: {len(leads_all)} | Assigned: {total_assigned} | Unassigned: {len(unassigned)}")